# loss-item-scalar-extract — worked example 2: Accumulate a Running Loss Using .item() in a Training Loop

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `loss-item-scalar-extract`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

A common training-loop pattern accumulates the loss over batches by adding `.item()` to a Python float accumulator. Because `.item()` breaks the connection to the autograd graph, the accumulated sum is just a number — it does not hold onto the computation graph for each step, so GPU memory can be freed between steps. Without `.item()`, assigning the raw tensor into a sum would keep every intermediate graph alive for the entire epoch.

## Worked solution

**Step 1 — set up a synthetic dataset.**
We create random targets and inputs with `torch.manual_seed` for reproducibility. The loop simulates 5 mini-batches, each producing an MSE-style loss tensor.

**Step 2 — accumulate with `.item()`.**
Inside the loop, `running_loss += loss.item()` adds a Python float to a Python float. After each step, the loss tensor goes out of scope and its graph can be freed. This is the canonical pattern — equivalent to what PyTorch Lightning logs internally.

**Step 3 — compute the epoch mean.**
After the loop, `epoch_loss = running_loss / n_steps` is a plain Python float division. No tensor arithmetic is involved at logging time.

**Step 4 — compare against keeping tensors.**
We also accumulate the raw tensors in a list to show that `.item()` produces the same numeric result — but the tensor-list approach would hold all graphs in memory until `.stack()` time.

In [ ]:
import torch as t

t.manual_seed(42)
n_steps = 5
running_loss = 0.0
tensor_accumulator = []  # for comparison only

for step in range(n_steps):
    t.manual_seed(step)  # reseed per step for reproducibility
    x = t.randn(8, 4)
    y = t.randn(8, 2)
    # Fake linear layer: random weights
    W = t.randn(2, 4, requires_grad=True)
    pred = x @ W.T
    loss = ((pred - y) ** 2).mean()

    # Accumulate as Python float — no graph retained
    running_loss += loss.item()
    tensor_accumulator.append(loss.detach().clone())

epoch_loss = running_loss / n_steps
tensor_mean = t.stack(tensor_accumulator).mean().item()

print(f"Epoch loss (item accumulation): {epoch_loss:.6f}")
print(f"Epoch loss (tensor accumulation): {tensor_mean:.6f}")
print(f"Agree within tolerance: {abs(epoch_loss - tensor_mean) < 1e-5}")
print(f"running_loss type: {type(running_loss).__name__}")